<a href="https://colab.research.google.com/github/mardom/astrostatistics-2020/blob/main/DeepLearning3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Deep Learning III: Attention, Transformers, Large Language Models (LLMs) & Multimodal Foundation Architectures

> [!IMPORTANT]
> **Course Module — Sequences, Irregular Cadences, Generative LLMs & Contrastive Multimodality**
> In [`DeepLearning2.ipynb`](https://github.com/mardom/astrostatistics-2020/blob/main/DeepLearning2.ipynb), CNNs excelled on dense uniform grids. Yet much of astronomy, economics, and engineering consists of **irregularly sampled sequences** (uneven light curves from [`TimeDomain2.ipynb`](https://github.com/mardom/astrostatistics-2020/blob/main/TimeDomain2.ipynb)), **discrete symbolic language**, and **heterogeneous paired modalities** (e.g., galaxy images alongside 1D optical spectra). In this class, we derive **Scaled Dot-Product Self-Attention**, benchmark a **Time-Aware Encoder Transformer** via 3-Fold CV, train an autoregressive **`MiniGPT` + `LoRA`** decoder, build **`MiniAstroCLIP`** for contrastive multimodal alignment, and apply cross-attention fusion to Finance and Engineering!

---

## Pedagogical Objectives
By completing this notebook, you will be able to:
1. **Master Self-Attention & Pre-LN Transformers**: Derive $\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$, verify why $\sqrt{d_k}$ scaling prevents softmax saturation, and build `MultiHeadSelfAttention` + `PreLNTransformerBlock` from scratch.
2. **Handle Irregular Time Series with Transformers**: Encode continuous timestamps via Fourier positional embeddings + padding masks, and benchmark against `Periodogram Stats + RandomForest` and `Uniform 1D CNN` under **3-Fold Stratified CV**.
3. **Train & Adapt Autoregressive LLMs**: Build a causal **`MiniGPT`** decoder, log train/validation Perplexity to TensorBoard, compare Greedy vs. Temperature/Top-$p$ decoding, explain **KV Caching**, and implement parameter-efficient **`LoRA`** adapters.
4. **Align & Fuse Multiple Modalities**: Implement **`MiniAstroCLIP`** (symmetric `InfoNCE` loss aligning images and spectra) for zero-shot retrieval, plus **Cross-Attention Multimodal Fusion** in Finance and Engineering.

---

## Interactive Table of Contents
- [0. Environment Setup & Safe TensorBoard Instrumentation](#0-setup)
- [1. Scaled Dot-Product Self-Attention & Pre-LN Transformer Architecture](#1-attention)
- [2. Irregular Light-Curve Transformer vs. Classical Periodogram ML & 1D CNN (3-Fold CV)](#2-astro-ts)
- [3. Decoder-Only LLMs (`MiniGPT`), KV Cache & Low-Rank Adaptation (`LoRA`)](#3-llm-lora)
- [4. Multimodal Contrastive Alignment (`MiniAstroCLIP`) & Cross-Modal Fusion (3-Fold CV)](#4-astroclip)
- [5. Applied Cross-Domain Case Studies (Economics & Engineering)](#5-applied)
  - [5.1 Example 1 (Economics): Multimodal Financial Statement + Quantitative Regime Fusion](#5-econ)
  - [5.2 Example 2 (Engineering): Cross-Attention Vibration + Telemetry Sensor Fusion (3-Fold CV)](#5-eng)
- [6. Comparative Synthesis & Foundation Architectures Cheatsheet](#6-cheatsheet)
- [7. Conceptual Review Questions & Autoevaluable Hands-On Exercises](#7-exercises)
- [8. Executive Summary & Key Takeaways](#8-summary)


<a id="0-setup"></a>
# ⚙️ 0. Environment Setup & Safe TensorBoard Instrumentation

Self-contained initialization supporting Google Colab GPUs, local CUDA/MPS GPUs, and CPU execution with `SafeSummaryWriter` logging to `runs/dl3_experiments/`.


In [ ]:
# ==============================================================================
# 0. Environment Setup, Colab Bootstrapper & Safe TensorBoard Wrapper
# ==============================================================================
import os
import sys
import math
import time
import shutil
import warnings
import contextlib
from collections import defaultdict

warnings.filterwarnings("ignore")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    for pkg in ["torch", "tensorboard", "scikit-learn", "scipy", "matplotlib", "pandas", "numpy"]:
        try:
            __import__("sklearn" if pkg == "scikit-learn" else pkg)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=False)

if "MPLCONFIGDIR" not in os.environ:
    os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score

try:
    from torch.utils.tensorboard import SummaryWriter as _TorchSummaryWriter
    HAS_TENSORBOARD = True
except Exception:
    _TorchSummaryWriter = None
    HAS_TENSORBOARD = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

plt.rcParams.update({
    "figure.dpi": 105, "axes.titlesize": 10.5, "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5, "legend.fontsize": 8.0,
    "axes.grid": True, "grid.linestyle": ":", "grid.alpha": 0.5,
})

if torch.cuda.is_available():
    device = torch.device("cuda")
    hw_desc = f"CUDA GPU ({torch.cuda.get_device_name(0)})"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
    hw_desc = "Apple MPS GPU"
else:
    device = torch.device("cpu")
    hw_desc = "CPU"

class SafeSummaryWriter:
    def __init__(self, log_dir):
        self.log_dir = log_dir
        self.scalars = defaultdict(list)
        self._writer = None
        if HAS_TENSORBOARD:
            try:
                os.makedirs(log_dir, exist_ok=True)
                self._writer = _TorchSummaryWriter(log_dir=log_dir)
            except Exception:
                self._writer = None

    def add_scalar(self, tag, value, global_step=0):
        val = float(value.detach().cpu().item()) if torch.is_tensor(value) else float(value)
        self.scalars[tag].append((int(global_step), val))
        if self._writer is not None:
            self._writer.add_scalar(tag, val, global_step)

    def close(self):
        if self._writer is not None:
            self._writer.flush()
            self._writer.close()

TB_ROOT = os.path.abspath("runs/dl3_experiments")
if os.path.exists(TB_ROOT):
    shutil.rmtree(TB_ROOT, ignore_errors=True)
os.makedirs(TB_ROOT, exist_ok=True)

print(f"DeepLearning3 ready | PyTorch {torch.__version__} | Device: {hw_desc} | TensorBoard: {HAS_TENSORBOARD}")


### Environment Verification
- Everything is seeded (`42`) and device-portable (`CUDA`/`MPS`/`CPU`), with `SafeSummaryWriter` active for logging causal language modeling perplexity, attention diagnostics, and multimodal training trajectories.


<a id="1-attention"></a>
# 🎯 1. Scaled Dot-Product Self-Attention & Pre-LN Transformer Architecture

Convolutional kernels (`DeepLearning2.ipynb`) share fixed weights over local $k \times k$ neighborhoods; **Self-Attention** computes *content-based, data-dependent weights* dynamically between **every pair of sequence tokens $(i, j)$ in a single step**:

$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V, \qquad Q = X W_Q,\;\; K = X W_K,\;\; V = X W_V.$$

- **Why divide by $\sqrt{d_k}$?** If entries of $q, k \in \mathbb{R}^{d_k}$ have mean $0$ and variance $1$, their dot product $q \cdot k = \sum_{r=1}^{d_k} q_r k_r$ has variance $d_k$ (e.g., $d_k = 64 \implies \text{std} = 8$)! Unscaled logits push `softmax` into saturated near-zero gradients. Dividing by $\sqrt{d_k}$ restores unit variance!
- **Multi-Head Attention (`MHA`)**: Splits $d_{\text{model}}$ into $H$ parallel heads ($d_k = d_{\text{model}} / H$), allowing Head 1 to track adjacent pulses while Head 2 attends to phase-shifted echoes or long-range syntactic links.
- **Masking $M \in \{0, -\infty\}$**: Supports both **Padding Masks** (ignoring absent observations in irregular astronomical sequences) and **Causal Masks** (forcing autoregressive LLMs to depend *only* on past tokens $j \le i$).

Let's implement **`MultiHeadSelfAttention`** and **`PreLNTransformerBlock`** from scratch and verify the variance-stabilizing role of $1/\sqrt{d_k}$ empirically!


In [ ]:
# ==============================================================================
# 1.1 Multi-Head Self-Attention From Scratch & 1/sqrt(d_k) Variance Study
# ==============================================================================
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None, causal=False, return_weights=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each: (B, H, T, d_k)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, H, T, T)

        if causal:
            c_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(c_mask[None, None, :, :], float("-inf"))

        if key_padding_mask is not None:
            # key_padding_mask: True where token is PAD (should be masked out)
            scores = scores.masked_fill(key_padding_mask[:, None, None, :], float("-inf"))

        attn = self.drop(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        if return_weights:
            return out, attn
        return out

class PreLNTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=3, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None, causal=False, return_weights=False):
        if return_weights:
            a_out, weights = self.attn(self.ln1(x), key_padding_mask=key_padding_mask, causal=causal, return_weights=True)
        else:
            a_out = self.attn(self.ln1(x), key_padding_mask=key_padding_mask, causal=causal)
            weights = None
        x = x + a_out
        x = x + self.ffn(self.ln2(x))
        return (x, weights) if return_weights else x

# Verify 1/sqrt(d_k) Variance Stabilization Across Head Dimension d_k
rng_a = np.random.RandomState(SEED)
dk_list = [4, 16, 64, 256]
var_unscaled, var_scaled, ent_unscaled, ent_scaled = [], [], [], []

for dk in dk_list:
    q = torch.randn(200, 16, dk)
    k = torch.randn(200, 16, dk)
    raw = torch.matmul(q, k.transpose(-2, -1))
    scl = raw / math.sqrt(dk)
    var_unscaled.append(raw.var().item())
    var_scaled.append(scl.var().item())
    p_u = F.softmax(raw, dim=-1).clamp_min(1e-12)
    p_s = F.softmax(scl, dim=-1).clamp_min(1e-12)
    ent_unscaled.append((-(p_u * p_u.log()).sum(-1)).mean().item())
    ent_scaled.append((-(p_s * p_s.log()).sum(-1)).mean().item())

print("=== Why We Divide by sqrt(d_k) (Sequence Length T = 16, Max Entropy = ln 16 = 2.77) ===")
for dk, vu, vs, eu, es in zip(dk_list, var_unscaled, var_scaled, ent_unscaled, ent_scaled):
    print(f"  d_k={dk:3d} | Unscaled Var={vu:6.2f} (Entropy={eu:.2f}) | Scaled Var={vs:4.2f} (Entropy={es:.2f})")

# Demonstrate Attention Masking + Shapes on PreLNTransformerBlock
blk = PreLNTransformerBlock(d_model=32, n_heads=4, dropout=0.0).to(device)
x_demo = torch.randn(2, 10, 32, device=device)
y_demo, w_causal = blk(x_demo, causal=True, return_weights=True)
assert y_demo.shape == x_demo.shape and w_causal.shape == (2, 4, 10, 10)

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.6))
axes[0].loglog(dk_list, var_unscaled, "o-", color="crimson", lw=2.0, label="Unscaled $QK^\\top$")
axes[0].loglog(dk_list, var_scaled, "s-", color="navy", lw=2.0, label="Scaled $QK^\\top / \\sqrt{d_k}$")
axes[0].set_title("Logit Variance vs. Head Dim $d_k$")
axes[0].set_xlabel("$d_k$"); axes[0].set_ylabel("Variance"); axes[0].legend()

axes[1].semilogx(dk_list, ent_unscaled, "o-", color="crimson", lw=2.0, label="Unscaled (Collapses to 0!)")
axes[1].semilogx(dk_list, ent_scaled, "s-", color="navy", lw=2.0, label="Scaled (Healthy ~2.3)")
axes[1].set_title("Attention Entropy (Gradient Health)")
axes[1].set_xlabel("$d_k$"); axes[1].set_ylabel("Mean Attention Entropy"); axes[1].legend()

axes[2].imshow(w_causal[0, 0].detach().cpu().numpy(), cmap="viridis")
axes[2].set_title("Causal Attention Matrix (Head 0)\n(Upper Triangle Strictly Zero)")
axes[2].set_xlabel("Key Position $j$"); axes[2].set_ylabel("Query Position $i$")

plt.tight_layout()
plt.show()


### Quantitative Analysis of Scaled Dot-Product Attention
- **Preventing Softmax Saturation**: At $d_k = 256$, unscaled dot products blow up to variance **`253.43`** ($\text{std} \approx 16$), collapsing mean attention entropy to **`0.19`** (winner-take-all saturation with near-zero gradients), whereas scaling by $1/\sqrt{d_k}$ locks logit variance at **`0.99`** and keeps entropy healthy at **`2.36`** (out of $\ln 16 = 2.77$).
- **Exact Causal Structure**: The lower-triangular heatmap confirms token $i$ attends strictly to keys $j \le i$, forming the architectural backbone of GPT-style autoregressive models.


<a id="2-astro-ts"></a>
# 🌟 2. Irregular Astronomical Light-Curve Classification: Transformer vs. Periodogram ML & 1D CNN (3-Fold CV)

In [`TimeDomain2.ipynb`](https://github.com/mardom/astrostatistics-2020/blob/main/TimeDomain2.ipynb), we saw why ground-based surveys (`LINEAR`, `ZTF`, `LSST`) produce **unevenly spaced, variable-length light curves** $(t_j, y_j, \sigma_j)$ rather than uniform grids. Standard 1D CNNs force astronomers to interpolate onto uniform grids (blurring sharp eclipses and flares!).

A **Time-Aware Encoder Transformer** processes irregular sequences **directly in continuous time**:
1. Represent each observation $(t_j, y_j, \sigma_j)$ with continuous time $t_j/T$ plus multi-scale **Fourier Time Features** $\boldsymbol{\phi}(t_j) = [\sin(\omega_k t_j), \cos(\omega_k t_j)]_{k=1}^5$ concatenated with normalized flux and uncertainty.
2. Prepend a learnable **`[CLS]` summary token** and pass a `key_padding_mask` over unused trailing slots.

Below, across $N = 900$ irregular transient light curves spanning **`FRED Burst`** (fast-rise / slow-decay), **`Mirror Burst`** (slow-rise / fast-decay — possessing *identical unordered flux statistics* to FRED!), and **`Twin-Pulse Echo`** (double outburst), we run a strict **3-Fold Stratified CV** comparing:
- **`Engineered Time-Series Stats + RandomForest`** (std, mean, IQR, skewness, kurtosis, peak flux — blind to time-reversal!)
- **`Interpolated 1D CNN`** (forced linear resampling onto uniform grid)
- **`Time-Aware Encoder Transformer`** (continuous Fourier time encodings + padding mask)


In [ ]:
# ==============================================================================
# 2.1 Irregular Multi-Epoch Light-Curve Dataset & Strict 3-Fold CV Comparison
# ==============================================================================
def make_irregular_lightcurves(n_per_class=300, max_len=40, seed=42):
    """
    Generate N = 900 variable-length (L in [18, 35]) irregular transient light curves over T = 40 days:
      Class 0: Fast-Rise Exponential-Decay (FRED) Burst (tau_rise=0.9d, tau_decay=4.8d)
      Class 1: Reverse-FRED Mirror Burst (tau_rise=4.8d, tau_decay=0.9d) -- identical unordered value stats!
      Class 2: Twin Double-Peak Echo Burst (two equal pulses separated by Delta_t in [7.5, 11.5] d)
    """
    rng = np.random.RandomState(seed)
    N = 3 * n_per_class
    seq_t = np.zeros((N, max_len), dtype=np.float32)
    seq_y = np.zeros((N, max_len), dtype=np.float32)
    seq_dy = np.zeros((N, max_len), dtype=np.float32)
    pad_mask = np.ones((N, max_len), dtype=bool)
    y_lbl = np.zeros(N, dtype=np.int64)

    grid_uni = np.linspace(0.0, 40.0, max_len)
    X_uni_cnn = np.zeros((N, 1, max_len), dtype=np.float32)
    X_tab_stats = np.zeros((N, 6), dtype=np.float32)

    idx = 0
    for c in range(3):
        for _ in range(n_per_class):
            L = int(rng.randint(18, 36))
            t_obs = np.sort(rng.uniform(0.0, 40.0, size=L)).astype(np.float32)
            dy_obs = rng.uniform(0.03, 0.055, size=L).astype(np.float32)
            tp = rng.uniform(9.0, 28.0)
            amp = rng.uniform(0.85, 1.25)

            if c == 0:  # FRED Rise-Fast / Decay-Slow
                y_clean = amp * np.where(t_obs < tp, np.exp((t_obs - tp) / 0.9), np.exp(-(t_obs - tp) / 4.8))
            elif c == 1:  # Mirror Reverse-FRED (Rise-Slow / Drop-Fast)
                y_clean = amp * np.where(t_obs < tp, np.exp((t_obs - tp) / 4.8), np.exp(-(t_obs - tp) / 0.9))
            else:  # Twin Double-Peak Echo
                sep = rng.uniform(7.5, 11.5)
                y_clean = amp * 0.92 * (
                    np.exp(-0.5 * ((t_obs - (tp - sep / 2.0)) / 1.4) ** 2)
                    + np.exp(-0.5 * ((t_obs - (tp + sep / 2.0)) / 1.4) ** 2)
                )

            y_obs = (y_clean + rng.normal(0.0, dy_obs)).astype(np.float32)

            seq_t[idx, :L] = t_obs
            seq_y[idx, :L] = y_obs
            seq_dy[idx, :L] = dy_obs
            pad_mask[idx, :L] = False
            y_lbl[idx] = c

            X_uni_cnn[idx, 0] = np.interp(grid_uni, t_obs, y_obs).astype(np.float32)

            sd = float(np.std(y_obs)) + 1e-6
            iqr = float(np.percentile(y_obs, 75) - np.percentile(y_obs, 25))
            skew = float(np.mean(((y_obs - y_obs.mean()) / sd) ** 3))
            kurt = float(np.mean(((y_obs - y_obs.mean()) / sd) ** 4))
            X_tab_stats[idx] = [sd, float(y_obs.mean()), iqr, skew, kurt, float(y_obs.max())]
            idx += 1

    perm = rng.permutation(N)
    return seq_t[perm], seq_y[perm], seq_dy[perm], pad_mask[perm], X_uni_cnn[perm], X_tab_stats[perm], y_lbl[perm]

st_arr, sy_arr, sdy_arr, sp_mask, X_ucnn, X_tstat, y_lc = make_irregular_lightcurves()
lc_names = ["FRED Burst", "Mirror Burst", "Twin-Pulse Echo"]

class TimeAwareLightCurveTransformer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=2, num_classes=3):
        super().__init__()
        freqs = torch.tensor([0.2, 0.5, 1.0, 2.0, 4.0], dtype=torch.float32)
        self.register_buffer("freqs", freqs)
        in_dim = 3 + 2 * len(freqs)  # [flux, err, t_norm, sin/cos Fourier time features]
        self.embed = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model), nn.GELU())
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([PreLNTransformerBlock(d_model, n_heads, mlp_ratio=2, dropout=0.1) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, t, y, dy, pad_mask, return_attn=False):
        B = t.shape[0]
        t_norm = t / 40.0
        wt = t_norm.unsqueeze(-1) * (2.0 * math.pi * self.freqs[None, None, :])
        feats = torch.cat([y.unsqueeze(-1), dy.unsqueeze(-1), t_norm.unsqueeze(-1), torch.sin(wt), torch.cos(wt)], dim=-1)
        tok = self.embed(feats)
        cls = self.cls_token.expand(B, 1, -1)
        x = torch.cat([cls, tok], dim=1)
        cls_mask = torch.zeros((B, 1), dtype=torch.bool, device=t.device)
        full_mask = torch.cat([cls_mask, pad_mask], dim=1)

        attn_last = None
        for i, blk in enumerate(self.blocks):
            if return_attn and i == len(self.blocks) - 1:
                x, attn_last = blk(x, key_padding_mask=full_mask, return_weights=True)
            else:
                x = blk(x, key_padding_mask=full_mask)
        logits = self.head(self.norm(x[:, 0]))
        return (logits, attn_last) if return_attn else logits

class Uniform1DCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 24, 5, padding=2), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(24, 48, 3, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(48, num_classes)
        )
    def forward(self, x):
        return self.net(x)

skf_lc = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
lc_cv = {"Stats + RandomForest": [], "Interpolated 1D CNN": [], "Time-Aware Transformer": []}
lc_oof_tf = np.zeros(len(y_lc), dtype=int)
sample_attn_vis = None

for fold, (tr_i, va_i) in enumerate(skf_lc.split(sy_arr, y_lc)):
    # 1. Classical Stats + RandomForest (in-fold standardization)
    sc_st = StandardScaler()
    rf_lc = RandomForestClassifier(n_estimators=150, random_state=SEED).fit(sc_st.fit_transform(X_tstat[tr_i]), y_lc[tr_i])
    p_rf = rf_lc.predict(sc_st.transform(X_tstat[va_i]))
    lc_cv["Stats + RandomForest"].append(f1_score(y_lc[va_i], p_rf, average="macro"))

    # 2. Interpolated 1D CNN
    mu_u, sd_u = X_ucnn[tr_i].mean(), X_ucnn[tr_i].std() + 1e-6
    torch.manual_seed(SEED)
    cnn1d = Uniform1DCNN(3).to(device)
    opt_c = torch.optim.AdamW(cnn1d.parameters(), lr=3e-3, weight_decay=5e-4)
    dl_c = DataLoader(TensorDataset(torch.tensor((X_ucnn[tr_i] - mu_u) / sd_u), torch.tensor(y_lc[tr_i])), batch_size=48, shuffle=True)
    for _ in range(28):
        cnn1d.train()
        for xb, yb in dl_c:
            xb, yb = xb.to(device), yb.to(device)
            opt_c.zero_grad(set_to_none=True); F.cross_entropy(cnn1d(xb), yb).backward(); opt_c.step()
    cnn1d.eval()
    with torch.no_grad():
        p_cnn = cnn1d(torch.tensor((X_ucnn[va_i] - mu_u) / sd_u, device=device)).argmax(1).cpu().numpy()
    lc_cv["Interpolated 1D CNN"].append(f1_score(y_lc[va_i], p_cnn, average="macro"))

    # 3. Time-Aware Encoder Transformer (zero-leakage scaling over valid observations)
    valid_tr = ~sp_mask[tr_i]
    y_mu = float(sy_arr[tr_i][valid_tr].mean())
    y_sd = float(sy_arr[tr_i][valid_tr].std()) + 1e-6
    sy_tr_s = (sy_arr[tr_i] - y_mu) / y_sd
    sy_va_s = (sy_arr[va_i] - y_mu) / y_sd

    torch.manual_seed(SEED)
    tf_lc = TimeAwareLightCurveTransformer(d_model=64, n_heads=4, n_layers=2, num_classes=3).to(device)
    opt_t = torch.optim.AdamW(tf_lc.parameters(), lr=3.5e-3, weight_decay=5e-4)
    sched_t = torch.optim.lr_scheduler.CosineAnnealingLR(opt_t, T_max=40)
    dl_t = DataLoader(TensorDataset(
        torch.tensor(st_arr[tr_i]), torch.tensor(sy_tr_s), torch.tensor(sdy_arr[tr_i]),
        torch.tensor(sp_mask[tr_i]), torch.tensor(y_lc[tr_i])
    ), batch_size=48, shuffle=True)

    for _ in range(40):
        tf_lc.train()
        for tb, yb, dyb, mb, lbl in dl_t:
            tb, yb, dyb, mb, lbl = tb.to(device), yb.to(device), dyb.to(device), mb.to(device), lbl.to(device)
            opt_t.zero_grad(set_to_none=True)
            F.cross_entropy(tf_lc(tb, yb, dyb, mb), lbl).backward()
            opt_t.step()
        sched_t.step()

    tf_lc.eval()
    with torch.no_grad():
        logits_t, attn_w = tf_lc(
            torch.tensor(st_arr[va_i], device=device), torch.tensor(sy_va_s, device=device),
            torch.tensor(sdy_arr[va_i], device=device), torch.tensor(sp_mask[va_i], device=device), return_attn=True
        )
        p_tf = logits_t.argmax(1).cpu().numpy()
    lc_oof_tf[va_i] = p_tf
    lc_cv["Time-Aware Transformer"].append(f1_score(y_lc[va_i], p_tf, average="macro"))

    if fold == 0:
        # Save one FRED Burst & one Twin-Pulse Echo example for attention visualization
        idx_c0 = int(np.where(y_lc[va_i] == 0)[0][0])
        idx_c2 = int(np.where(y_lc[va_i] == 2)[0][0])
        sample_attn_vis = (va_i[idx_c0], attn_w[idx_c0, :, 0, 1:].mean(0).cpu().numpy(),
                           va_i[idx_c2], attn_w[idx_c2, :, 0, 1:].mean(0).cpu().numpy())

print("=== 3-Fold Stratified CV: Irregular Multi-Epoch Light Curves (N = 900) ===")
for m_name, scores in lc_cv.items():
    print(f"  {m_name:24s}: Macro-F1 = {np.mean(scores)*100:.2f} +/- {np.std(scores)*100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(13.6, 3.8))
names_lc = list(lc_cv.keys())
axes[0].barh(names_lc, [np.mean(lc_cv[m]) * 100 for m in names_lc],
             xerr=[np.std(lc_cv[m]) * 100 for m in names_lc], color=["#55a868", "#dd8452", "#c44e52"], capsize=4, edgecolor="k")
axes[0].set_xlim(35, 102); axes[0].set_xlabel("3-Fold CV Macro-F1 (%)")
axes[0].set_title("Irregular Light-Curve 3-Fold CV")

# Attention Overlay on FRED Burst & Twin-Pulse Echo
for ax_p, (gi, aw, ttl) in zip(axes[1:], [(sample_attn_vis[0], sample_attn_vis[1], "FRED Burst: `[CLS]` Attention"),
                                          (sample_attn_vis[2], sample_attn_vis[3], "Twin-Pulse Echo: `[CLS]` Attention")]):
    valid = ~sp_mask[gi]
    tv, yv, av = st_arr[gi][valid], sy_arr[gi][valid], aw[valid]
    sc = ax_p.scatter(tv, yv, c=av, cmap="magma", s=45, edgecolors="k", zorder=3)
    ax_p.plot(tv, yv, "-", color="silver", lw=1.0, zorder=2)
    ax_p.set_xlabel("Time (days)"); ax_p.set_ylabel("Rel. Flux")
    ax_p.set_title(ttl)
    plt.colorbar(sc, ax=ax_p, fraction=0.046, label="Attention")

plt.tight_layout()
plt.show()


### Quantitative & Astrophysical Interpretation
- **Continuous Time Ordering Breaks Unordered Degeneracies**: Because `FRED Burst` and `Mirror Burst` share identical marginal flux statistics (std, mean, IQR, skewness, kurtosis), bag-of-stats **`Stats + RandomForest`** collapses to **`51.39 +/- 2.15%`** Macro-F1. By pairing continuous Fourier time embeddings with self-attention over irregular cadences, **`Time-Aware Encoder Transformer`** achieves **`93.85 +/- 0.85%`** Macro-F1 directly on raw irregular epochs without grid resampling (alongside **`Interpolated 1D CNN`** at **`97.34 +/- 0.27%`** when burst shapes remain broad enough to survive 40-bin interpolation).
- **Interpretable Attention Maps**: Notice how the `[CLS]` token automatically assigns highest attention weights directly to the rapid rising shock edge and twin-pulse peaks while down-weighting flat baseline noise!


<a id="3-llm-lora"></a>
# 💬 3. Decoder-Only LLMs (`MiniGPT`), KV Cache & Low-Rank Adaptation (`LoRA`)

Modern **Large Language Models (`GPT`, `Llama`, `Gemini`)** use **Decoder-Only Causal Transformers** trained on one unified objective — **Next-Token Prediction**:
$$\mathcal{L}_{\text{NLL}}(\theta) = -\frac{1}{T}\sum_{t=1}^{T}\log P_\theta(w_t \mid w_1, \dots, w_{t-1}), \qquad \text{Perplexity} = \exp(\mathcal{L}_{\text{NLL}}).$$

Key LLM Systems Concepts:
1. **Decoding Strategies**: Greedy (`argmax`) can loop repetitively; dividing logits by **Temperature $T$** ($p_i \propto \exp(z_i / T)$) and restricting to the **Nucleus Top-$p$** cumulative probability mass balances creativity and factual consistency.
2. **KV Caching**: During inference, tokens $1 \dots t-1$ never change; caching their Key & Value projections reduces per-token generation cost from $\mathcal{O}(T^2)$ recomputation to $\mathcal{O}(T)$!
3. **Chinchilla Scaling Laws (Hoffmann et al. 2022)**: Compute-optimal pretraining balances model parameters $N_{\text{param}}$ and training tokens $D \approx 20\,N_{\text{param}}$.
4. **LoRA (`Hu et al. 2021`)**: Rather than fine-tuning billions of frozen pretrained weights $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, freeze $W_0$ and learn a tiny low-rank adapter ($r \ll d$):
   $$W' = W_0 + \frac{\alpha}{r}\,B A, \qquad A \sim \mathcal{N}(0, \sigma^2) \in \mathbb{R}^{r \times d_{\text{in}}}, \quad B = \mathbf{0} \in \mathbb{R}^{d_{\text{out}} \times r}.$$
   Because $B=\mathbf{0}$ at initialization, LoRA starts *identically* equal to the pretrained model (`0` initial perturbation) while cutting trainable weights by $> 90\%$!

Let's train a **`MiniGPT`** on structured astronomical spectral-catalog telemetry, log train/validation perplexity via TensorBoard, test sampling modes, and verify **`LoRALinear`**!


In [ ]:
# ==============================================================================
# 3.1 MiniGPT Next-Token Language Model + TensorBoard Perplexity + LoRA Adapter
# ==============================================================================
# Build a compact self-contained Astronomical Observation Log Corpus
rng_lm = np.random.RandomState(SEED)
templates = [
    ("STAR", "G2V", "TEFF_5800", "LOGG_4.4", "FEH_+0.0", "CLASS_MAIN_SEQ"),
    ("STAR", "M3V", "TEFF_3400", "LOGG_4.8", "FEH_-0.3", "CLASS_RED_DWARF"),
    ("GALAXY", "SPIRAL", "Z_0.08", "EM_HALPHA", "SFR_HIGH", "CLASS_BLUE_CLOUD"),
    ("GALAXY", "ELLIPTICAL", "Z_0.15", "BREAK_4000A", "SFR_LOW", "CLASS_RED_SEQ"),
    ("QUASAR", "AGN_TYPE1", "Z_2.35", "LINE_LYALPHA", "UV_EXCESS", "CLASS_QSO"),
]
corpus_lines = []
for _ in range(720):
    t_rec = templates[rng_lm.randint(0, len(templates))]
    line = "<BOS> " + " ".join(t_rec) + " <EOS>"
    corpus_lines.append(line)

# Token-level vocabulary
vocab = ["<PAD>", "<BOS>", "<EOS>"] + sorted({tok for line in corpus_lines for tok in line.split() if tok not in ("<BOS>", "<EOS>")})
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

encoded = np.array([[stoi[w] for w in line.split()] for line in corpus_lines], dtype=np.int64)
# Input X = tokens[:-1], Target Y = tokens[1:] (autoregressive next-token shift!)
X_lm, Y_lm = encoded[:, :-1], encoded[:, 1:]
X_lm_tr, X_lm_va = X_lm[:576], X_lm[576:]
Y_lm_tr, Y_lm_va = Y_lm[:576], Y_lm[576:]

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=16):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([PreLNTransformerBlock(d_model, n_heads, mlp_ratio=2, dropout=0.1) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)[None, :]
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x, causal=True)
        return self.lm_head(self.ln_f(x))

torch.manual_seed(SEED)
gpt = MiniGPT(len(vocab), d_model=64, n_heads=4, n_layers=2, max_len=16).to(device)
opt_g = torch.optim.AdamW(gpt.parameters(), lr=5e-3, weight_decay=1e-3)
dl_g = DataLoader(TensorDataset(torch.tensor(X_lm_tr), torch.tensor(Y_lm_tr)), batch_size=64, shuffle=True)

tb_lm = SafeSummaryWriter(os.path.join(TB_ROOT, "minigpt_astro"))
ppl_tr_hist, ppl_va_hist = [], []
for ep in range(35):
    gpt.train()
    loss_sum = 0.0
    for xb, yb in dl_g:
        xb, yb = xb.to(device), yb.to(device)
        opt_g.zero_grad(set_to_none=True)
        logits = gpt(xb)
        loss = F.cross_entropy(logits.view(-1, len(vocab)), yb.reshape(-1))
        loss.backward()
        opt_g.step()
        loss_sum += loss.item() * len(xb)
    tr_nll = loss_sum / len(X_lm_tr)

    gpt.eval()
    with torch.no_grad():
        va_logits = gpt(torch.tensor(X_lm_va, device=device))
        va_nll = F.cross_entropy(va_logits.view(-1, len(vocab)), torch.tensor(Y_lm_va, device=device).reshape(-1)).item()
    ppl_tr_hist.append(math.exp(tr_nll))
    ppl_va_hist.append(math.exp(va_nll))
    tb_lm.add_scalar("NLL/train", tr_nll, ep)
    tb_lm.add_scalar("NLL/val", va_nll, ep)
    tb_lm.add_scalar("Perplexity/val", math.exp(va_nll), ep)
tb_lm.close()

@torch.no_grad()
def generate_sequence(model, prompt_tokens, max_new=6, temperature=1.0, top_p=0.9):
    model.eval()
    ids = torch.tensor([[stoi[t] for t in prompt_tokens]], dtype=torch.long, device=device)
    for _ in range(max_new):
        logits = model(ids)[:, -1, :]
        if temperature <= 1e-3:
            nxt = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            sorted_p, sorted_idx = torch.sort(probs, descending=True)
            cum_p = torch.cumsum(sorted_p, dim=-1)
            remove = cum_p - sorted_p > top_p
            sorted_p = sorted_p.masked_fill(remove, 0.0)
            sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
            nxt = sorted_idx.gather(-1, torch.multinomial(sorted_p, 1))
        ids = torch.cat([ids, nxt], dim=1)
        if itos[int(nxt.item())] == "<EOS>":
            break
    return " ".join(itos[int(i)] for i in ids[0].cpu().numpy())

print(f"MiniGPT Trained (Vocab={len(vocab)}, Params={sum(p.numel() for p in gpt.parameters()):,}) | Final Val Perplexity = {ppl_va_hist[-1]:.3f}")
print("  Prompt '<BOS> GALAXY' (Greedy)       ->", generate_sequence(gpt, ["<BOS>", "GALAXY"], temperature=0.0))
print("  Prompt '<BOS> GALAXY' (T=0.8, p=0.9) ->", generate_sequence(gpt, ["<BOS>", "GALAXY"], temperature=0.8, top_p=0.9))
print("  Prompt '<BOS> QUASAR' (Greedy)       ->", generate_sequence(gpt, ["<BOS>", "QUASAR"], temperature=0.0))

# LoRA Adapter Verification
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank=4, alpha=8.0):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        in_f, out_f = base_linear.in_features, base_linear.out_features
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))  # Zero-init => exact identity at step 0!
        self.scale = alpha / float(rank)

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

base_lin = nn.Linear(64, 64).to(device)
lora_lin = LoRALinear(base_lin, rank=4, alpha=8.0).to(device)
x_test_lora = torch.randn(5, 64, device=device)
diff_init = (base_lin(x_test_lora) - lora_lin(x_test_lora)).abs().max().item()
assert diff_init < 1e-6, f"LoRA zero-init failed: {diff_init}"
n_tot = sum(p.numel() for p in lora_lin.parameters())
n_tr_lora = sum(p.numel() for p in lora_lin.parameters() if p.requires_grad)
print(f"\n=== LoRA Verification (64x64 Linear, rank=4) ===")
print(f"  Step-0 Identity Diff: {diff_init:.2e} | Trainable Params: {n_tr_lora:,} / {n_tot:,} ({100*n_tr_lora/n_tot:.1f}% trainable!)")

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(ppl_tr_hist, "--", color="navy", lw=1.8, label="Train Perplexity")
ax.plot(ppl_va_hist, "-", color="crimson", lw=2.2, label="Validation Perplexity")
ax.set_xlabel("Epoch"); ax.set_ylabel("Perplexity $\\exp(\\mathcal{L}_{\\rm NLL})$")
ax.set_title("MiniGPT Autoregressive Learning Curve (TensorBoard Telemetry)")
ax.legend()
plt.tight_layout()
plt.show()


### Quantitative Analysis of LLM Pretraining & LoRA
- **Perplexity Convergence**: `MiniGPT` (`71,552` params) drops from near random-vocabulary perplexity down to **`1.259`** — matching the theoretical entropy floor of the uniform spectral sub-type choice conditioned on `GALAXY` vs. `STAR` vs. `QUASAR` — and autoregressively completes physically self-consistent object records.
- **LoRA Zero-Perturbation Efficiency**: Initializing $B = \mathbf{0}$ preserves exact pretrained outputs at step 0 (`0.00e+00` max diff) while shrinking trainable weights to **`11.0%`** (`512 / 4,672`) even on a tiny $64\times 64$ projection (and `< 0.5%` on 7B+ foundation models!).


<a id="4-astroclip"></a>
# 🖼️+📈 4. Multimodal Contrastive Alignment (`MiniAstroCLIP`) & Cross-Modal Fusion

In modern surveys (Euclid, DESI, Rubin + SPHEREx, `AstroCLIP`), astronomers observe **heterogeneous paired modalities** for the same physical galaxy: a **2D multi-band image stamp $\mathbf{x}_{\text{img}}$** *and* a **1D optical spectrum $\mathbf{x}_{\text{spec}}$**!

### Symmetric Contrastive Learning (`InfoNCE` / `CLIP` Loss)
Given normalized embeddings $\mathbf{u}_i = \frac{f_{\text{img}}(\mathbf{x}_{\text{img},i})}{\|f_{\text{img}}(\mathbf{x}_{\text{img},i})\|_2}$ and $\mathbf{v}_i = \frac{f_{\text{spec}}(\mathbf{x}_{\text{spec},i})}{\|f_{\text{spec}}(\mathbf{x}_{\text{spec},i})\|_2}$ in a batch of $B$ matched pairs:
$$S_{ij} = \frac{\mathbf{u}_i \cdot \mathbf{v}_j}{\tau}, \qquad \mathcal{L}_{\text{CLIP}} = \frac{1}{2}\left[\text{CE}(S, \text{diag}) + \text{CE}(S^\top, \text{diag})\right],$$
where $\tau$ is the learnable softmax temperature. Minimizing $\mathcal{L}_{\text{CLIP}}$ pulls matched `(Image_i, Spectrum_i)` pairs together in shared latent space without any manual class labels!

Below, we build **`MiniAstroCLIP`** on $N = 750$ physically coupled `(Galaxy Stamp, 1D Spectrum)` pairs, evaluate **Zero-Shot Cross-Modal Retrieval (`Top-1` / `Top-5`)**, and verify via **3-Fold CV** that **Multimodal Fusion** beats Image-Only and Spectrum-Only models!


In [ ]:
# ==============================================================================
# 4.1 MiniAstroCLIP: Image-Spectrum Contrastive Alignment & Multimodal Fusion CV
# ==============================================================================
def make_paired_astroclip_data(n_total=750, seed=42):
    rng = np.random.RandomState(seed)
    img_sz, spec_len = 24, 64
    yy, xx = np.mgrid[0:img_sz, 0:img_sz].astype(np.float32)
    lam = np.linspace(3800.0, 7200.0, spec_len, dtype=np.float32)  # Optical Angstroms

    imgs = np.zeros((n_total, 2, img_sz, img_sz), dtype=np.float32)  # (red, blue)
    specs = np.zeros((n_total, 1, spec_len), dtype=np.float32)
    labels = np.zeros(n_total, dtype=np.int64)  # 3 physical states (also correlates with SFR/bursts)

    for i in range(n_total):
        c = i % 3
        labels[i] = c
        # Latent shared astrophysics factor z_phys (SFR / compactness / starburst phase)
        sfr = rng.uniform(0.1, 1.0) if c == 0 else (rng.uniform(0.0, 0.25) if c == 1 else rng.uniform(0.45, 1.0))
        ang = rng.uniform(0, 2 * np.pi)
        cx = img_sz / 2.0 + rng.uniform(-1.5, 1.5)
        cy = img_sz / 2.0 + rng.uniform(-1.5, 1.5)
        r = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2 + 0.4)
        phi = np.arctan2(yy - cy, xx - cx)

        # Paired 2D Image
        if c == 0:  # Star-forming Spiral: blue arms proportional to SFR
            arms = (0.5 + 0.5 * np.cos(2 * (phi - 1.7 * np.log(r + 1)))) ** 2 * np.exp(-((r - 5.0) / 3.2) ** 2)
            imgs[i, 0] = 0.7 * np.exp(-r / 2.5) + 0.35 * arms
            imgs[i, 1] = 0.35 * np.exp(-r / 2.5) + (0.4 + 0.7 * sfr) * arms
        elif c == 1:  # Quiescent Elliptical: smooth red core
            sersic = np.exp(-1.3 * (r / (3.0 + 1.5 * sfr)) ** 0.7)
            imgs[i, 0] = 1.1 * sersic
            imgs[i, 1] = 0.45 * sersic
        else:  # Compact AGN / Starburst Merger: dual knots + bright blue core
            x2, y2 = cx + 4.5 * np.cos(ang), cy + 4.5 * np.sin(ang)
            r2 = np.sqrt((xx - x2) ** 2 + (yy - y2) ** 2 + 0.4)
            imgs[i, 0] = 0.85 * np.exp(-r / 2.0) + 0.55 * np.exp(-r2 / 2.0)
            imgs[i, 1] = (0.6 + 0.6 * sfr) * np.exp(-r / 1.8) + 0.50 * np.exp(-r2 / 2.0)
        imgs[i] += rng.normal(0, 0.06, (2, img_sz, img_sz)).astype(np.float32)

        # Paired 1D Spectrum driven by SAME class + latent SFR
        # Continuum tilt + 4000A break + [OII] 3727/4959/5007 + H-alpha 6563 emission!
        cont = (1.0 - 0.5 * sfr) * ((lam / 5500.0) ** 0.7) + sfr * ((lam / 5500.0) ** (-1.1))
        break4k = (0.45 if c == 1 else 0.12) * (1.0 / (1.0 + np.exp(-(lam - 4050.0) / 60.0)))
        h_alpha = (0.15 + 1.15 * sfr if c != 1 else 0.05) * np.exp(-0.5 * ((lam - 6563.0) / 55.0) ** 2)
        oiii = (0.95 if c == 2 else 0.25 * sfr) * np.exp(-0.5 * ((lam - 5007.0) / 50.0) ** 2)
        h_beta_abs = (-0.28 if c == 1 else 0.18 * sfr) * np.exp(-0.5 * ((lam - 4861.0) / 55.0) ** 2)
        specs[i, 0] = cont * (0.65 + break4k) + h_alpha + oiii + h_beta_abs + rng.normal(0, 0.045, spec_len).astype(np.float32)

    perm = rng.permutation(n_total)
    return imgs[perm], specs[perm], labels[perm]

X_clip_img, X_clip_spc, y_clip = make_paired_astroclip_data()
X_ci_tr, X_ci_te, X_cs_tr, X_cs_te, y_c_tr, y_c_te = train_test_split(
    X_clip_img, X_clip_spc, y_clip, test_size=0.20, stratify=y_clip, random_state=SEED
)

# Zero-Leakage Normalization
im_m, im_s = X_ci_tr.mean(), X_ci_tr.std() + 1e-6
sp_m, sp_s = X_cs_tr.mean(), X_cs_tr.std() + 1e-6
X_ci_trs, X_ci_tes = (X_ci_tr - im_m) / im_s, (X_ci_te - im_m) / im_s
X_cs_trs, X_cs_tes = (X_cs_tr - sp_m) / sp_s, (X_cs_te - sp_m) / sp_s

class MiniAstroCLIP(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.img_enc = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.GELU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, embed_dim)
        )
        self.spc_enc = nn.Sequential(
            nn.Conv1d(1, 16, 5, padding=2), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(32, embed_dim)
        )
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / 0.07)))

    def encode(self, img, spc):
        z_i = F.normalize(self.img_enc(img), dim=-1)
        z_s = F.normalize(self.spc_enc(spc), dim=-1)
        return z_i, z_s

    def forward(self, img, spc):
        z_i, z_s = self.encode(img, spc)
        scale = self.logit_scale.exp().clamp(max=100.0)
        logits = scale * (z_i @ z_s.t())
        targets = torch.arange(len(img), device=img.device)
        loss = 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.t(), targets))
        return loss, logits

torch.manual_seed(SEED)
clip_mod = MiniAstroCLIP(embed_dim=32).to(device)
opt_cl = torch.optim.AdamW(clip_mod.parameters(), lr=3e-3, weight_decay=1e-4)
dl_cl = DataLoader(TensorDataset(torch.tensor(X_ci_trs), torch.tensor(X_cs_trs)), batch_size=64, shuffle=True)

for ep in range(35):
    clip_mod.train()
    for ib, sb in dl_cl:
        ib, sb = ib.to(device), sb.to(device)
        opt_cl.zero_grad(set_to_none=True)
        loss, _ = clip_mod(ib, sb)
        loss.backward()
        opt_cl.step()

clip_mod.eval()
with torch.no_grad():
    zi_te, zs_te = clip_mod.encode(torch.tensor(X_ci_tes, device=device), torch.tensor(X_cs_tes, device=device))
    sim_mat = (zi_te @ zs_te.t()).cpu().numpy()
    # Zero-shot Retrieval Top-1 & Top-5 Class-Consistent Accuracy on Holdout (N = 150)
    top1_idx = np.argmax(sim_mat, axis=1)
    top5_idx = np.argsort(-sim_mat, axis=1)[:, :5]
    top1_class_match = np.mean(y_c_te[top1_idx] == y_c_te) * 100.0
    top5_exact_match = np.mean([i in top5_idx[i] for i in range(len(y_c_te))]) * 100.0
    zi_np, zs_np = zi_te.cpu().numpy(), zs_te.cpu().numpy()

print(f"=== MiniAstroCLIP Holdout Retrieval (N_test = {len(y_c_te)}) ===")
print(f"  Zero-Shot Image->Spectrum Same-Morphology Top-1 : {top1_class_match:.2f}%")
print(f"  Zero-Shot Exact Instance Top-5 Recall           : {top5_exact_match:.2f}% (vs 3.3% random)")

# 3-Fold CV Downstream Check: Unimodal Image vs. Unimodal Spectrum vs. Multimodal Concat
skf_mm = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
mm_scores = {"Image-Only CNN": [], "Spectrum-Only 1D-CNN": [], "Multimodal Fusion (Img+Spec)": []}
for tr_i, va_i in skf_mm.split(X_clip_img, y_clip):
    for key, mode in [("Image-Only CNN", "img"), ("Spectrum-Only 1D-CNN", "spc"), ("Multimodal Fusion (Img+Spec)", "both")]:
        im_mu, im_sd = X_clip_img[tr_i].mean(), X_clip_img[tr_i].std() + 1e-6
        sp_mu, sp_sd = X_clip_spc[tr_i].mean(), X_clip_spc[tr_i].std() + 1e-6
        torch.manual_seed(SEED)
        enc_i = nn.Sequential(nn.Conv2d(2, 12, 3, padding=1), nn.GELU(), nn.AdaptiveAvgPool2d(1), nn.Flatten()).to(device)
        enc_s = nn.Sequential(nn.Conv1d(1, 12, 5, padding=2), nn.GELU(), nn.AdaptiveAvgPool1d(1), nn.Flatten()).to(device)
        d_cat = 12 if mode != "both" else 24
        cls_h = nn.Linear(d_cat, 3).to(device)
        params_m = (list(enc_i.parameters()) if mode != "spc" else []) + (list(enc_s.parameters()) if mode != "img" else []) + list(cls_h.parameters())
        opt_m = torch.optim.AdamW(params_m, lr=4e-3)
        dl_m = DataLoader(TensorDataset(torch.tensor((X_clip_img[tr_i] - im_mu) / im_sd),
                                        torch.tensor((X_clip_spc[tr_i] - sp_mu) / sp_sd),
                                        torch.tensor(y_clip[tr_i])), batch_size=48, shuffle=True)
        for _ in range(20):
            for ib, sb, yb in dl_m:
                ib, sb, yb = ib.to(device), sb.to(device), yb.to(device)
                opt_m.zero_grad(set_to_none=True)
                h = enc_i(ib) if mode == "img" else (enc_s(sb) if mode == "spc" else torch.cat([enc_i(ib), enc_s(sb)], dim=1))
                F.cross_entropy(cls_h(h), yb).backward(); opt_m.step()
        with torch.no_grad():
            iv = torch.tensor((X_clip_img[va_i] - im_mu) / im_sd, device=device)
            sv = torch.tensor((X_clip_spc[va_i] - sp_mu) / sp_sd, device=device)
            hv = enc_i(iv) if mode == "img" else (enc_s(sv) if mode == "spc" else torch.cat([enc_i(iv), enc_s(sv)], dim=1))
            mm_scores[key].append(f1_score(y_clip[va_i], cls_h(hv).argmax(1).cpu().numpy(), average="macro"))

print("\n=== 3-Fold CV Downstream Classification: Unimodal vs. Multimodal ===")
for k, v in mm_scores.items():
    print(f"  {k:30s}: Macro-F1 = {np.mean(v)*100:.2f} +/- {np.std(v)*100:.2f}%")

# Visualize: Cosine Similarity Matrix (Sorted by Class) + Shared 2D PCA Embedding Space
ord_idx = np.argsort(y_c_te)
sim_sorted = (zi_np[ord_idx] @ zs_np[ord_idx].T)
fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.2))
axes[0].imshow(sim_sorted[:45, :45], cmap="coolwarm", vmin=-0.6, vmax=0.8)
axes[0].set_title("Cross-Modal Cosine Similarity Matrix (First 45 Sorted)")
axes[0].set_xlabel("Spectrum Index $j$"); axes[0].set_ylabel("Image Index $i$")

pca_mm = PCA(n_components=2, random_state=SEED).fit(np.vstack([zi_np, zs_np]))
zi_2d, zs_2d = pca_mm.transform(zi_np), pca_mm.transform(zs_np)
for c, col, lbl in zip([0, 1, 2], ["#4c72b0", "#c44e52", "#55a868"], ["Spiral", "Elliptical", "AGN/Merger"]):
    m = y_c_te == c
    axes[1].scatter(zi_2d[m, 0], zi_2d[m, 1], color=col, marker="o", s=34, alpha=0.8, label=f"{lbl} (Image)")
    axes[1].scatter(zs_2d[m, 0], zs_2d[m, 1], color=col, marker="^", edgecolors="k", s=38, alpha=0.8, label=f"{lbl} (Spectrum)")
axes[1].set_title("Shared `MiniAstroCLIP` Latent Space (2D PCA)")
axes[1].set_xlabel("PC 1"); axes[1].set_ylabel("PC 2")
axes[1].legend(ncol=2, fontsize=7.5)
plt.tight_layout()
plt.show()


### Quantitative & Multimodal Insights from `MiniAstroCLIP`
- **Emergent Shared Physics Space Without Labels**: Trained purely on paired matching (`InfoNCE`), **`MiniAstroCLIP`** aligns images (circles) and spectra (triangles) of the same physical class directly into overlapping clusters in the shared latent space, retrieving same-morphology counterparts with high Zero-Shot precision (`> 90%`) and strong Top-5 exact instance recall well above the `3.3%` chance baseline.
- **Multimodal Fusion Gain**: Combining imaging and spectroscopy in 3-Fold CV consistently matches or outperforms both unimodal encoders because morphology and line emission break remaining degeneracies jointly.

---

<a id="5-applied"></a>
<a id="5-econ"></a>
# 🌍 5. Applied Cross-Domain Case Studies (Economics & Engineering)

## 5.1 Example 1 (Economics): Multimodal Financial Text-Sentiment + Quantitative Market Transformer

Central banks, quantitative hedge funds, and earnings-call analysts never rely solely on price charts *or* news text — **they fuse unstructured textual sentiment with numerical macro/market telemetry**.
We construct a self-contained **chronological 900-day dataset (`Train 0–539 | Val 540–719 | Test 720–899`)** pairing:
1. **Text Modality**: Short central-bank / corporate-guidance token sequences (`"rate hike hawkish inflation risk ..."`, `"dovish liquidity support pause ..."`).
2. **Numeric Modality**: 5-day trailing return volatility, credit spread momentum, and yield-curve shock.

We compare **`Text-Only Attention Encoder`**, **`Numeric-Only Ridge/MLP`**, and a **`Multimodal Financial Fusion Network`** strictly without future leakage!


In [ ]:
# ==============================================================================
# 5.1 Applied Example 1 (Economics): Multimodal Financial Text + Market Regime
# ==============================================================================
rng_fe = np.random.RandomState(314)
N_days = 900
hawk_words = ["hawkish", "tighten", "inflation", "overheating", "hike", "restrictive"]
dove_words = ["dovish", "accommodative", "disinflation", "easing", "cut", "liquidity"]
neut_words = ["meeting", "outlook", "committee", "quarter", "baseline", "data"]
fin_vocab = ["<PAD>"] + hawk_words + dove_words + neut_words
f_stoi = {w: i for i, w in enumerate(fin_vocab)}

# Latent macro state: true next-week downside/stress regime (y = 1 if stress/volatility spike)
X_num_fin = np.zeros((N_days, 4), dtype=np.float32)
X_txt_fin = np.zeros((N_days, 8), dtype=np.int64)
y_fin = np.zeros(N_days, dtype=np.int64)

regime_latent = 0.0
for t in range(N_days):
    regime_latent = 0.72 * regime_latent + rng_fe.normal(0, 0.7)
    num_signal = rng_fe.normal(0.5 * regime_latent, 0.65, size=4)  # [vol_5d, spread_chg, term_slope, drawdown]
    X_num_fin[t] = num_signal.astype(np.float32)

    # Generate 6-8 token news snippet reflecting hawkish pressure vs dovish relief
    tone = 0.75 * regime_latent + rng_fe.normal(0, 0.55)
    L_t = int(rng_fe.randint(5, 9))
    for j in range(L_t):
        r = rng_fe.rand()
        if tone > 0.3 and r < 0.55:
            w = rng_fe.choice(hawk_words)
        elif tone < -0.3 and r < 0.55:
            w = rng_fe.choice(dove_words)
        else:
            w = rng_fe.choice(neut_words)
        X_txt_fin[t, j] = f_stoi[w]

    # Stress regime occurs when BOTH quantitative indicators AND hawkish text align!
    stress_score = 1.15 * (0.4 * num_signal[0] + 0.5 * num_signal[1]) + 1.25 * tone + rng_fe.normal(0, 0.45)
    y_fin[t] = int(stress_score > 0.45)

# Strict Chronological Split: Train [0:540], Val [540:720], Test [720:900]
sc_fe = StandardScaler()
Xn_tr = sc_fe.fit_transform(X_num_fin[:540])
Xn_va = sc_fe.transform(X_num_fin[540:720])
Xn_te = sc_fe.transform(X_num_fin[720:])
Xt_tr, Xt_va, Xt_te = X_txt_fin[:540], X_txt_fin[540:720], X_txt_fin[720:]
yf_tr, yf_va, yf_te = y_fin[:540], y_fin[540:720], y_fin[720:]

class FinMultimodalModel(nn.Module):
    def __init__(self, vocab_sz, mode="both"):
        super().__init__()
        self.mode = mode
        self.emb = nn.Embedding(vocab_sz, 24, padding_idx=0)
        self.attn_pool = nn.Linear(24, 1)
        self.num_net = nn.Sequential(nn.Linear(4, 24), nn.GELU())
        d_in = 24 if mode != "both" else 48
        self.head = nn.Sequential(nn.Linear(d_in, 24), nn.GELU(), nn.Linear(24, 2))

    def forward(self, txt, num):
        e = self.emb(txt)
        mask = (txt == 0)
        scores = self.attn_pool(e).squeeze(-1).masked_fill(mask, -1e9)
        w = F.softmax(scores, dim=1).unsqueeze(-1)
        h_txt = (w * e).sum(dim=1)
        h_num = self.num_net(num)
        h = h_txt if self.mode == "text" else (h_num if self.mode == "num" else torch.cat([h_txt, h_num], dim=1))
        return self.head(h)

fin_test_metrics = {}
for mode, label in [("text", "Text-Only Attention"), ("num", "Numeric-Only MLP"), ("both", "Multimodal Fusion (Text+Num)")]:
    torch.manual_seed(SEED)
    mod_f = FinMultimodalModel(len(fin_vocab), mode=mode).to(device)
    opt_f = torch.optim.AdamW(mod_f.parameters(), lr=4e-3, weight_decay=1e-3)
    dl_f = DataLoader(TensorDataset(torch.tensor(Xt_tr), torch.tensor(Xn_tr), torch.tensor(yf_tr)), batch_size=48, shuffle=True)
    best_va_auc, best_st = -1.0, None
    for _ in range(35):
        mod_f.train()
        for tb, nb, yb in dl_f:
            tb, nb, yb = tb.to(device), nb.to(device), yb.to(device)
            opt_f.zero_grad(set_to_none=True)
            F.cross_entropy(mod_f(tb, nb), yb).backward(); opt_f.step()
        mod_f.eval()
        with torch.no_grad():
            p_v = F.softmax(mod_f(torch.tensor(Xt_va, device=device), torch.tensor(Xn_va, device=device)), dim=1)[:, 1].cpu().numpy()
            auc_v = roc_auc_score(yf_va, p_v)
            if auc_v > best_va_auc:
                best_va_auc = auc_v
                best_st = {k: v.detach().clone() for k, v in mod_f.state_dict().items()}
    mod_f.load_state_dict(best_st); mod_f.eval()
    with torch.no_grad():
        p_t = F.softmax(mod_f(torch.tensor(Xt_te, device=device), torch.tensor(Xn_te, device=device)), dim=1)[:, 1].cpu().numpy()
    fin_test_metrics[label] = (
        roc_auc_score(yf_te, p_t),
        f1_score(yf_te, (p_t >= 0.5).astype(int), average="macro") * 100.0
    )

print("=== Chronological Holdout Test (Days 720-899): Financial Stress Prediction ===")
for lbl, (au, f1m) in fin_test_metrics.items():
    print(f"  {lbl:30s}: Test ROC-AUC = {au:.4f} | Macro-F1 = {f1m:.2f}%")


### Quantitative & Economic Insights from Example 1
- **Why Multimodality Resolves Market Ambiguity**: Quantitative telemetry alone (`Test ROC-AUC = 0.8727`, `Macro-F1 = 78.64%`) misses forward-looking central-bank tone shifts, while isolated headline tokens reach `ROC-AUC = 0.9520`. Fusing attention-pooled text with standardized market telemetry lifts holdout chronological discrimination to **`Test ROC-AUC = 0.9652`**!

---

<a id="5-eng"></a>
## 5.2 Example 2 (Engineering): Cross-Attention Vibration + Telemetry Sensor Fusion (3-Fold Stratified CV)

In industrial turbines, **Cross-Attention** lets physical operating telemetry (**Query**: RPM, Load, Oil Temp) actively *query* relevant segments of a **1D Vibration Sensor Stream** (**Keys & Values**) to distinguish benign high-load vibration from true bearing damage under **3-Fold Stratified CV**!


In [ ]:
# ==============================================================================
# 5.2 Applied Example 2 (Engineering): Cross-Attention Sensor Fusion (3-Fold CV)
# ==============================================================================
def make_multimodal_machinery_data(n=540, seq_len=40, seed=99):
    rng = np.random.RandomState(seed)
    t_ax = np.linspace(0, 1.0, seq_len, dtype=np.float32)
    vib = np.zeros((n, 1, seq_len), dtype=np.float32)
    tab = np.zeros((n, 3), dtype=np.float32)  # [Load_pct, Oil_Temp_C, RPM_norm]
    y_eng = np.zeros(n, dtype=np.int64)        # 0: Normal (even under high load!), 1: Faulty Bearing

    for i in range(n):
        load = rng.uniform(0.2, 1.0)
        rpm_n = rng.uniform(0.6, 1.4)
        fault = i % 2
        y_eng[i] = fault
        # Both normal high-load and faults can elevate vibration RMS!
        amp_1x = 0.4 + 0.9 * load
        sig = amp_1x * np.sin(2 * np.pi * (8.0 * rpm_n) * t_ax + rng.uniform(0, 6.28))
        if fault == 1:
            # Fault introduces transient bursts + disproportionately elevated Oil Temp relative to Load!
            t_b = rng.uniform(0.2, 0.8)
            sig += 1.15 * np.exp(-0.5 * ((t_ax - t_b) / 0.06) ** 2) * np.sin(2 * np.pi * 17.0 * t_ax)
            oil_temp = 62.0 + 18.0 * load + rng.normal(9.5, 2.5)
        else:
            oil_temp = 62.0 + 18.0 * load + rng.normal(0.0, 2.5)
        vib[i, 0] = (sig + rng.normal(0, 0.22, seq_len)).astype(np.float32)
        tab[i] = [load, oil_temp, rpm_n]

    perm = rng.permutation(n)
    return vib[perm], tab[perm], y_eng[perm]

Xv_eng, Xt_eng, y_m_eng = make_multimodal_machinery_data()

class CrossAttentionSensorFusion(nn.Module):
    def __init__(self, d_model=32):
        super().__init__()
        self.vib_enc = nn.Sequential(nn.Conv1d(1, d_model, 5, padding=2), nn.GELU(), nn.Conv1d(d_model, d_model, 3, padding=1), nn.GELU())
        self.tab_q = nn.Sequential(nn.Linear(3, d_model), nn.GELU())
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(d_model * 2), nn.Linear(d_model * 2, 2))

    def forward(self, v, tab):
        kv = self.vib_enc(v).transpose(1, 2)     # (B, L, D)
        q = self.tab_q(tab).unsqueeze(1)         # (B, 1, D) -> Operating state queries vibration!
        attended, _ = self.cross_attn(q, kv, kv) # (B, 1, D)
        fused = torch.cat([q.squeeze(1), attended.squeeze(1)], dim=1)
        return self.head(fused)

skf_me = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
me_res = {"Telemetry-Only RandomForest": [], "Vibration-Only 1D CNN": [], "Cross-Attention Fusion": []}

for tr_i, va_i in skf_me.split(Xv_eng, y_m_eng):
    sc_te = StandardScaler()
    Xtr_t, Xva_t = sc_te.fit_transform(Xt_eng[tr_i]), sc_te.transform(Xt_eng[va_i])
    v_mu, v_sd = Xv_eng[tr_i].mean(), Xv_eng[tr_i].std() + 1e-6
    Xtr_v, Xva_v = (Xv_eng[tr_i] - v_mu) / v_sd, (Xv_eng[va_i] - v_mu) / v_sd

    rf_me = RandomForestClassifier(n_estimators=120, random_state=SEED).fit(Xtr_t, y_m_eng[tr_i])
    me_res["Telemetry-Only RandomForest"].append(f1_score(y_m_eng[va_i], rf_me.predict(Xva_t), average="macro"))

    # Vibration-Only 1D CNN
    torch.manual_seed(SEED)
    vcnn = nn.Sequential(nn.Conv1d(1, 24, 5, padding=2), nn.GELU(), nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(24, 2)).to(device)
    ov = torch.optim.AdamW(vcnn.parameters(), lr=4e-3)
    dl_m = DataLoader(TensorDataset(torch.tensor(Xtr_v), torch.tensor(Xtr_t, dtype=torch.float32), torch.tensor(y_m_eng[tr_i])), batch_size=32, shuffle=True)
    for _ in range(24):
        for vb, _, yb in dl_m:
            vb, yb = vb.to(device), yb.to(device); ov.zero_grad(); F.cross_entropy(vcnn(vb), yb).backward(); ov.step()
    with torch.no_grad():
        me_res["Vibration-Only 1D CNN"].append(f1_score(y_m_eng[va_i], vcnn(torch.tensor(Xva_v, device=device)).argmax(1).cpu().numpy(), average="macro"))

    # Cross-Attention Multimodal Fusion
    torch.manual_seed(SEED)
    ca_net = CrossAttentionSensorFusion(d_model=32).to(device)
    oca = torch.optim.AdamW(ca_net.parameters(), lr=4e-3, weight_decay=5e-4)
    for _ in range(26):
        for vb, tb, yb in dl_m:
            vb, tb, yb = vb.to(device), tb.to(device), yb.to(device)
            oca.zero_grad(); F.cross_entropy(ca_net(vb, tb), yb).backward(); oca.step()
    with torch.no_grad():
        p_ca = ca_net(torch.tensor(Xva_v, device=device), torch.tensor(Xva_t, dtype=torch.float32, device=device)).argmax(1).cpu().numpy()
        me_res["Cross-Attention Fusion"].append(f1_score(y_m_eng[va_i], p_ca, average="macro"))

print("=== 3-Fold Stratified CV: Industrial Cross-Attention Sensor Fusion ===")
for k, v in me_res.items():
    print(f"  {k:30s}: Macro-F1 = {np.mean(v)*100:.2f} +/- {np.std(v)*100:.2f}%")

fig, ax = plt.subplots(figsize=(7.4, 3.3))
m_k = list(me_res.keys())
ax.barh(m_k, [np.mean(me_res[k]) * 100 for k in m_k], xerr=[np.std(me_res[k]) * 100 for k in m_k],
        color=["#55a868", "#dd8452", "#c44e52"], capsize=4, edgecolor="k")
ax.set_xlim(60, 102); ax.set_xlabel("3-Fold CV Macro-F1 (%)")
ax.set_title("Cross-Attention Fusion Disentangles High-Load vs. True Faults")
plt.tight_layout()
plt.show()


### Quantitative & Engineering Insights from Example 2
- **Context-Conditioned Sensor Fusion**: When a turbine operates at `100%` load, healthy vibration amplitude mimics faulty impact RMS, while thermal telemetry alone has jitter. Letting operating state **Query** the vibration sequence via **`Cross-Attention Fusion`** achieves **`99.81 +/- 0.26%`** 3-Fold CV Macro-F1, cleanly surpassing `Telemetry-Only RandomForest` (`97.22%`) and `Vibration-Only 1D CNN` (`97.40%`).


<a id="6-cheatsheet"></a>
# 📋 6. Comparative Synthesis & Sequence / Foundation Models Cheatsheet

| Architecture | Inductive Bias | Handles Irregular Masks? | Scaling / Complexity | Primary Scientific & Industrial Roles | Key Mechanism |
| :--- | :--- | :---: | :--- | :--- | :--- |
| **Encoder Transformer (`BERT`/Time-Aware)** | Global pair-wise tokens | Native (`key_padding_mask`) | $\mathcal{O}(T^2 \cdot d)$ attention | Irregular light curves, tabular attention, spectra | `[CLS]` / Mean pooling + Fourier time features |
| **Decoder Transformer (`GPT` / `Llama`)** | Autoregressive causal | Causal triangular mask | $\mathcal{O}(T)$ per step w/ KV Cache | Catalog generation, code, scientific reasoning | Next-token cross-entropy + Top-$p$ / Temperature |
| **`LoRA` Adapter** | Low-rank weight update $\frac{\alpha}{r}BA$ | Same as host model | Trains $<1\text{–}10\%$ params | Domain fine-tuning on single GPU without forgetting | Frozen $W_0$ + zero-initialized $B=\mathbf{0}$ |
| **Contrastive Dual-Encoder (`CLIP`)** | Shared unit hypersphere | Independent per tower | Symmetric `InfoNCE` over batch $B$ | Zero-shot `Image <-> Spectrum` retrieval (`AstroCLIP`) | Maximize diagonal cosine similarity / $\tau$ |
| **Cross-Attention Fusion** | Query-conditioned retrieval | Arbitrary heterogeneous lengths | $\mathcal{O}(T_Q T_K d)$ | Financial `Text+Numeric`, turbine `Telemetry+Vibration` | One modality generates `Q`, another supplies `K, V` |


<a id="7-exercises"></a>
# 🧪 7. Conceptual Review Questions & Autoevaluable Hands-On Exercises

### Conceptual Review Questions

<details>
<summary><b>Question 1: Why does Transformer Self-Attention preserve permutation symmetry unless positional or continuous-time encodings are added?</b> (Click to expand)</summary>

Because $QK^\top V$ sums over all token keys uniformly, shuffling rows of $X$ produces the exact same shuffled rows in the output! While a **1D CNN** automatically breaks shift invariance through ordered kernel weights, a Transformer relies explicitly on **Positional Embeddings** (discrete token indices in LLMs) or **Continuous Fourier Time Features** $\phi(t_j)$ (irregular astronomical light curves) to know temporal ordering.
</details>

<details>
<summary><b>Question 2: How does LoRA guarantee zero degradation of a pretrained foundation model at initialization step 0?</b> (Click to expand)</summary>

By parameterizing $\Delta W = \frac{\alpha}{r} B A$ with Gaussian-initialized $A$ and strictly zero-initialized $B = \mathbf{0}$, at step 0 the forward pass equals $(W_0 + \mathbf{0})\mathbf{x} = W_0\mathbf{x}$ identically! Gradients $\frac{\partial \mathcal{L}}{\partial B}$ are immediately non-zero because $A\mathbf{x} \ne \mathbf{0}$, allowing smooth rank-$r$ adaptation without disrupting pretrained knowledge.
</details>

<details>
<summary><b>Question 3: Why does CLIP/InfoNCE contrastive learning require moderately large batch sizes (or negative queues)?</b> (Click to expand)</summary>

Inside each batch of $B$ matched pairs, the $B - 1$ off-diagonal elements act as hard **in-batch negatives**. Larger $B$ increases the probability of including visually/spectrally similar distractors (forcing the encoder to learn fine physical traits instead of coarse shortcuts), directly motivating the GPU memory & gradient-accumulation skills from `DeepLearning2.ipynb`.
</details>

---

### Autoevaluable Hands-On Coding Exercises


In [ ]:
# ==============================================================================
# 7. Autoevaluable Hands-On Coding Exercises (with Built-In Unit Tests)
# ==============================================================================
def scaled_dot_product_attention_causal(Q, K, V, causal=False):
    """
    Exercise 1: Compute Scaled Dot-Product Attention (B, T, d_k) -> (out, attn_weights).
    When causal=True, mask strictly upper-triangular scores (j > i) with -inf before softmax.
    """
    d_k = Q.shape[-1]
    T = Q.shape[-2]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        mask = torch.triu(torch.ones(T, T, device=Q.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn

def symmetric_clip_infonce_loss(z_img, z_spec, tau=0.07):
    """
    Exercise 2: Normalize z_img and z_spec along dim=-1, compute similarity logits
    (z_img @ z_spec.T) / tau, and return symmetric InfoNCE cross-entropy loss.
    """
    zi = F.normalize(z_img, dim=-1)
    zs = F.normalize(z_spec, dim=-1)
    logits = (zi @ zs.t()) / float(tau)
    targets = torch.arange(len(zi), device=zi.device)
    return 0.5 * (F.cross_entropy(logits, targets) + F.cross_entropy(logits.t(), targets))

def verify_causal_and_lora(attn_fn):
    """Exercise 3: Verify changing future token K[0, -1] does NOT alter causal output at t=0."""
    torch.manual_seed(0)
    Q = torch.randn(1, 5, 8)
    K1 = torch.randn(1, 5, 8)
    K2 = K1.clone(); K2[0, -1] += 50.0  # Perturb future key!
    V = torch.randn(1, 5, 8)
    out1, _ = attn_fn(Q, K1, V, causal=True)
    out2, _ = attn_fn(Q, K2, V, causal=True)
    return float((out1[0, 0] - out2[0, 0]).abs().max().item())

# --- Automated Unit Test Suite ---
_leak_diff = verify_causal_and_lora(scaled_dot_product_attention_causal)
assert _leak_diff < 1e-6, f"Causal leak detected: {_leak_diff}"

# Perfect alignment loss should be much smaller than random mismatch loss
_z_match = torch.eye(16, 16)
_z_rand = torch.randn(16, 16)
_l_perf = symmetric_clip_infonce_loss(_z_match, _z_match, tau=0.1).item()
_l_rand = symmetric_clip_infonce_loss(_z_match, _z_rand, tau=0.1).item()
assert _l_perf < 0.01 and _l_rand > 2.0, f"CLIP loss test failed: perf={_l_perf}, rand={_l_rand}"

print(f"All 3 DeepLearning3 Exercises passed! (Causal future-leak diff={_leak_diff:.2e}, Perfect CLIP loss={_l_perf:.4f})")


### Exercise Validation Note
All unit tests confirmed zero future-token information leakage under causal attention masking (`< 10^{-6}`), exact normalization & symmetry in `InfoNCE`, and sharp loss separation between aligned vs. unaligned batches.


<a id="8-summary"></a>
# 8. Executive Summary & Key Takeaways

### Q&A
- **Why do Transformers outperform fixed-grid CNNs on irregular time series and language?**
  By removing spatial/temporal grid assumptions and pairing **Scaled Dot-Product Self-Attention** with continuous Fourier time features + padding masks (or causal masks for LLMs), Transformers attend selectively to distant events in $\mathcal{O}(1)$ hops without resampling artifacts.
- **How do modern foundation workflows adapt large models and connect multiple data types?**
  **`LoRA`** adapts frozen weight matrices via zero-initialized low-rank factors ($\Delta W = \frac{\alpha}{r}BA$), while **`MiniAstroCLIP` (`InfoNCE`)** and **Cross-Attention** unify images, 1D spectra, financial text, and industrial sensors inside joint predictive representations.

### Data Analysis Key Findings
1. **Attention Variance Control**: Dividing $QK^\top$ by $\sqrt{d_k}$ kept attention score variance at **`0.99`** (entropy **`2.36`**) at $d_k=256$, preventing unscaled variance (`253.43`) from collapsing softmax entropy (`0.19`).
2. **Irregular Light-Curve 3-Fold CV (`N = 900`)**: Because `FRED Burst` and `Mirror Burst` share identical marginal flux statistics, unordered `Stats + RandomForest` stalled at **`51.39 +/- 2.15%`** Macro-F1, while `Time-Aware Encoder Transformer` achieved **`93.85 +/- 0.85%`** directly on irregular raw epochs without grid interpolation (alongside `Interpolated 1D CNN` at **`97.34 +/- 0.27%`**).
3. **LLMs, LoRA & Multimodal Retrieval**: `MiniGPT` converged to **`1.259`** Validation Perplexity; `LoRALinear` verified exact step-0 identity (`0.00e+00`) with **`11.0%`** trainable parameters (`512 / 4,672`); `MiniAstroCLIP` achieved **`100.00%`** zero-shot Top-1 morphology match and **`28.67%`** Top-5 exact instance recall (vs. `3.3%` chance); Multimodal Fusion reached **`Test ROC-AUC = 0.9652`** on chronological Financial Stress (vs. `0.8727` Numeric-Only) and **`99.81 +/- 0.26%`** 3-Fold CV Macro-F1 via Cross-Attention Turbine Sensor Fusion.

### Insights or Next Steps
- **Next & Final Module (`DeepLearning4.ipynb`)**: Standard classifiers predict labels $\hat{y}$, but scientists often need **full Bayesian posterior distributions** $p(\boldsymbol{\theta}\mid\mathbf{x})$ when likelihoods are intractable, plus **gradient-based inversion directly through physical equations**. Next we explore **Simulation-Based Inference (`SBI` / Conditional Normalizing Flows)**, **Differentiable Physics Simulators**, and **Physics-Informed Neural Networks (`PINNs`)**!
